# NB03 — Final Hybrid Static RAG Retrieval

This notebook defines the **final frozen Build02 retriever**.

## Final Build02 architecture

**Regulation library**  
→ **legal-aware chunks**  
→ **Dense semantic retrieval**  
+ **BM25 lexical retrieval**  
→ **RRF hybrid ranking**  
→ **source diversification**  
→ **fixed Top-5 evidence**  
→ NB04 LLM generation

### Why this is the final Build02 retriever

Earlier experiments showed:

- Dense-only retrieval was too weak for several legal questions.
- Hybrid retrieval improved retrieval by combining semantic similarity with exact lexical matching.
- Multi-query retrieval improved some questions but was inconsistent and started moving toward query-planning / agent behavior.

Therefore Build02 is frozen as a **single-query, single-shot hybrid static RAG**.

### Important boundary

This notebook does **not**:
- rewrite or decompose the benchmark question;
- use Verified Reference answers for ranking;
- inject article numbers or expected numeric answers;
- call an LLM;
- retry retrieval based on missing evidence.

Each benchmark question is searched **once**, and the final Top-5 is frozen.


In [2]:
# Run once if needed
# %pip install -q -U sentence-transformers rank-bm25 pandas numpy


In [3]:
from pathlib import Path
import json
import re
import hashlib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from IPython.display import display

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

LEGAL_PATH = DATA_DIR / "chunks_legal_aware.json"
BENCHMARK_PATH = DATA_DIR / "benchmark_questions_v1.json"

OUTPUT_PATH = DATA_DIR / "retrieval_results_nb03_final_hybrid_static_rag.csv"
CONFIG_PATH = DATA_DIR / "retrieval_config_nb03_final_hybrid_static_rag.json"

for p in [LEGAL_PATH, BENCHMARK_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Missing required file: {p}")

legal_chunks = json.loads(LEGAL_PATH.read_text(encoding="utf-8"))
benchmark = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
questions = benchmark["questions"]

assert list(questions) == [f"Q{i}" for i in range(1, 11)]
assert all(isinstance(q, str) and q.strip() for q in questions.values())

BENCHMARK_SHA256 = hashlib.sha256(
    json.dumps(
        questions,
        ensure_ascii=False,
        sort_keys=True
    ).encode("utf-8")
).hexdigest()

print("Benchmark:", benchmark["benchmark_id"])
print("Legal-aware chunks:", len(legal_chunks))
print("Benchmark SHA256:", BENCHMARK_SHA256)


Benchmark: regulatory_agent_v1_notion_2026-10-08
Legal-aware chunks: 1383
Benchmark SHA256: 2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a1a6f62f1ec849c3e4e


## 1. QA guard for the legal-aware library

The earlier NB02 fix grouped the Türkiye dried-fig Annex II entry into one logical chunk.

That chunk should contain:

- Türkiye
- dried figs
- aflatoxins
- ochratoxin A
- 30% identity/physical check frequency

This guard only checks that the expected NB02 output is present.  
It does **not** change ranking.


In [4]:
turkiye_fig_chunks = [
    c for c in legal_chunks
    if "2019_1793" in str(c.get("file", ""))
    and "türkiye" in str(c.get("text", "")).casefold()
    and "dried figs" in str(c.get("text", "")).casefold()
    and "aflatoxin" in str(c.get("text", "")).casefold()
    and "ochratoxin" in str(c.get("text", "")).casefold()
    and "30%" in str(c.get("text", ""))
]

print("Complete Türkiye dried-fig logical chunks:", len(turkiye_fig_chunks))

if len(turkiye_fig_chunks) != 1:
    raise AssertionError(
        "Expected exactly one complete Türkiye dried-fig logical chunk. "
        "Rerun the final NB02 legal-aware chunking notebook first."
    )

print("PASS: NB02 logical row-group fix is present.")


Complete Türkiye dried-fig logical chunks: 1
PASS: NB02 logical row-group fix is present.


## 2. Build searchable legal text

The dense and lexical retrievers search the same representation of each chunk.

We include legal metadata because fields such as regulation title, article and legal location can be important retrieval signals.


In [5]:
def safe(value):
    return "" if value is None else str(value)

def searchable_text(chunk):
    parts = [
        safe(chunk.get("regulation_title")),
        safe(chunk.get("file")),
        safe(chunk.get("location")),
        safe(chunk.get("article")),
        safe(chunk.get("heading")),
        safe(chunk.get("text")),
    ]
    return " | ".join(p for p in parts if p).strip()

legal_texts = [searchable_text(c) for c in legal_chunks]

assert len(legal_texts) == len(legal_chunks)
assert all(legal_texts)

print("Example searchable chunk:")
print(legal_texts[0][:1200])


Example searchable chunk:
COMMISSION REGULATION (EU) 2023/915 | EU_2023_915_consolidated_2026-08-19.html | Article 1 — Definitions | Article 1 | Definitions | Article 1 Definitions For the purposes of this Regulation, the following definitions apply: (a) ‘food’ means food as defined in Article 2 of Regulation (EC) No 178/2002 of the European Parliament and of the Council ( 1 ); (b) ‘food business operator’ means food business operator as defined in Article 3, point 3, of Regulation (EC) No 178/2002; (c) ‘placing on the market’ means placing on the market as defined in Article 3, point 8, of Regulation (EC) No 178/2002; (d) ‘final consumer’ means final consumer as defined in Article 3, point 18, of Regulation (EC) No 178/2002; (e) ‘processing’ means processing as defined in Article 2(1), point (m) of Regulation (EC) No 852/2004 of the European Parliament and of the Council ( 2 ); (f) ‘unprocessed products’ means unprocessed products as defined in Article 2(1), point (n), of Regulation (

## 3. Dense semantic retrieval

**Semantic retrieval** asks:

> Which chunks are closest in meaning to the question?

The same multilingual SentenceTransformer model used in the earlier benchmark is kept here so that the retrieval change is attributable to the hybrid method rather than a new embedding model.

Embeddings are normalized, therefore the dot product is equivalent to cosine similarity.


In [6]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

model = SentenceTransformer(EMBEDDING_MODEL)

legal_embeddings = model.encode(
    legal_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

query_ids = list(questions.keys())
query_texts = [questions[qid] for qid in query_ids]

query_embeddings = model.encode(
    query_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

dense_rank_by_qid = {}
dense_score_by_qid = {}

for i, qid in enumerate(query_ids):
    scores = legal_embeddings @ query_embeddings[i]
    order = np.argsort(scores)[::-1]

    dense_score_by_qid[qid] = scores
    dense_rank_by_qid[qid] = order

print("Dense semantic retrieval ready.")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Dense semantic retrieval ready.


## 4. BM25 lexical retrieval

**Lexical retrieval** asks:

> Which chunks contain the same important words or legal tokens as the question?

BM25 complements semantic search by rewarding exact matches such as:

- product names,
- contaminant names,
- legal terminology,
- numbers,
- article-like tokens.

No benchmark-specific keywords are added manually.


In [7]:
TOKEN_RE = re.compile(r"[A-Za-zÀ-ÿ0-9]+(?:[.,][0-9]+)?%?")

def tokenize(text):
    return TOKEN_RE.findall(text.casefold())

tokenized_corpus = [tokenize(text) for text in legal_texts]
bm25 = BM25Okapi(tokenized_corpus)

bm25_rank_by_qid = {}
bm25_score_by_qid = {}

for qid in query_ids:
    scores = np.asarray(
        bm25.get_scores(tokenize(questions[qid])),
        dtype=float
    )
    order = np.argsort(scores)[::-1]

    bm25_score_by_qid[qid] = scores
    bm25_rank_by_qid[qid] = order

print("BM25 lexical retrieval ready.")


BM25 lexical retrieval ready.


## 5. RRF hybrid ranking

Dense cosine scores and BM25 scores are on different numerical scales, so we do not add the raw scores directly.

Instead we use **Reciprocal Rank Fusion (RRF)**:

- a chunk ranked highly by dense retrieval receives credit;
- a chunk ranked highly by BM25 receives credit;
- a chunk ranked highly by both receives more combined credit.

Only the rank positions are fused.


In [8]:
RRF_K = 60
CANDIDATE_DEPTH = 50

def rrf_fuse(qid, candidate_depth=CANDIDATE_DEPTH, rrf_k=RRF_K):
    dense_order = dense_rank_by_qid[qid][:candidate_depth]
    bm25_order = bm25_rank_by_qid[qid][:candidate_depth]

    fused_scores = {}

    for rank, idx in enumerate(dense_order, start=1):
        idx = int(idx)
        fused_scores[idx] = (
            fused_scores.get(idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    for rank, idx in enumerate(bm25_order, start=1):
        idx = int(idx)
        fused_scores[idx] = (
            fused_scores.get(idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    fused_order = sorted(
        fused_scores,
        key=fused_scores.get,
        reverse=True
    )

    return np.asarray(fused_order, dtype=int), fused_scores


hybrid_rank_by_qid = {}
hybrid_score_by_qid = {}

for qid in query_ids:
    order, scores = rrf_fuse(qid)
    hybrid_rank_by_qid[qid] = order
    hybrid_score_by_qid[qid] = scores

print("Hybrid RRF ranking ready.")


Hybrid RRF ranking ready.


## 6. Source diversification

A Top-5 list can otherwise be dominated by several near-duplicate chunks from the same legal unit.

The diversification rule is generic:

> maximum 2 chunks from the same `file + article/location` group.

No regulation or article is forced into the result.


In [9]:
MAX_PER_SOURCE_GROUP = 2
TOP_K = 5

def source_group(chunk):
    file_name = safe(chunk.get("file"))
    article = safe(chunk.get("article"))
    location = safe(chunk.get("location"))

    legal_unit = article if article else location
    return (file_name, legal_unit)


def diversified_topk(
    order,
    k=TOP_K,
    max_per_source_group=MAX_PER_SOURCE_GROUP
):
    selected = []
    counts = {}

    for idx in order:
        idx = int(idx)
        chunk = legal_chunks[idx]
        group = source_group(chunk)

        if counts.get(group, 0) >= max_per_source_group:
            continue

        selected.append(idx)
        counts[group] = counts.get(group, 0) + 1

        if len(selected) == k:
            break

    return selected

print("Diversification rule ready.")


Diversification rule ready.


## 7. Freeze the final Top-5 evidence for Q1–Q10

This is the final Build02 retrieval step.

Each official benchmark question is passed to the retriever **once**.

The resulting Top-5 is the frozen evidence that NB04 should receive.


In [10]:
rows = []

for qid in query_ids:
    selected = diversified_topk(
        hybrid_rank_by_qid[qid],
        k=TOP_K
    )

    assert len(selected) == TOP_K

    for rank, idx in enumerate(selected, start=1):
        chunk = legal_chunks[idx]

        rows.append({
            "qid": qid,
            "question": questions[qid],
            "benchmark_id": benchmark["benchmark_id"],
            "benchmark_sha256": BENCHMARK_SHA256,

            # Keep 'legal_aware' for easy compatibility with NB04 filtering.
            "strategy": "legal_aware",

            # Explicit final retrieval method.
            "retrieval_method": "hybrid_dense_bm25_rrf_diverse_v1",

            "rank": rank,
            "score": float(
                hybrid_score_by_qid[qid].get(idx, 0.0)
            ),

            "dense_cosine_score": float(
                dense_score_by_qid[qid][idx]
            ),
            "bm25_score": float(
                bm25_score_by_qid[qid][idx]
            ),

            "chunk_id": chunk.get("chunk_id", ""),
            "file": chunk.get("file", ""),
            "version_date": chunk.get("version_date", ""),
            "location": chunk.get("location", ""),
            "article": chunk.get("article", ""),
            "heading": chunk.get("heading", ""),
            "text": chunk.get("text", ""),
        })

retrieval_df = pd.DataFrame(rows)

assert len(retrieval_df) == 50
assert retrieval_df.groupby("qid").size().eq(5).all()
assert retrieval_df["qid"].nunique() == 10
assert retrieval_df["chunk_id"].astype(bool).all()

display(
    retrieval_df[
        [
            "qid",
            "rank",
            "file",
            "location",
            "article",
            "score",
        ]
    ]
)


,qid,rank,file,location,article,score
0,Q1,1,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",NaN,0.032787
1,Q1,2,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN,0.028787
2,Q1,3,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 14",NaN,0.027047
3,Q1,4,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 12",NaN,0.026743
4,Q1,5,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 10",NaN,0.025978
5,Q2,1,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 9",NaN,0.030798
6,Q2,2,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 13",NaN,0.030366
7,Q2,3,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 11",NaN,0.030310
8,Q2,4,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 14",NaN,0.030090
9,Q2,5,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 6",NaN,0.029052


## 8. Human-readable Top-5 inspection

This cell is for qualitative review only.

It does not change the frozen ranking.


In [11]:
for qid in query_ids:
    print("\n" + "=" * 110)
    print(qid, "—", questions[qid])

    qdf = retrieval_df[
        retrieval_df["qid"] == qid
    ].sort_values("rank")

    for _, row in qdf.iterrows():
        preview = re.sub(
            r"\s+",
            " ",
            safe(row["text"])
        )[:450]

        print(
            f'\n#{int(row["rank"])} | '
            f'{row["file"]} | '
            f'{row["location"]} | '
            f'{row["article"]}'
        )
        print(preview)



Q1 — A shipment of dried figs originating from Türkiye is found to contain aflatoxin B1. Which EU legislation is relevant to this case?

#1 | EU_2019_1793_consolidated_2026-06-30.html | Annex II, Section 1, Row 14 — Türkiye dried-fig products | nan
COMMISSION IMPLEMENTING REGULATION (EU) 2019/1793. Legal location: Annex II, Section 1, logical row 14 — Türkiye (TR). Product scope: dried figs; mixtures of nuts or dried fruits containing figs; dried fig paste; dried figs prepared or preserved, including mixtures. Hazards listed for this logical entry: aflatoxins and ochratoxin A. Frequency of identity and physical checks: 30%. Table column context: Row | Country of origin | Food and feed (inte

#2 | EU_2023_915_consolidated_2026-08-19.html | Table 3, row 6 | nan
COMMISSION REGULATION (EU) 2023/915. Table context: 1 | Mycotoxins || 1.1 | Aflatoxins | Maximum level (μg/kg) | Remarks || B 1 | Sum of B 1 , B 2 , G 1 and G 2 | M 1 | For the sum of aflatoxins, maximum levels refer to lower bou

## 9. Save the frozen retrieval and configuration

The retrieval CSV is the evidence file NB04 needs.

The configuration JSON records the exact static retrieval design so the experiment can be reproduced later.


In [12]:
retrieval_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

config = {
    "build": "Build02",
    "retrieval_type": "frozen_static_rag",
    "query_mode": "single_query_single_shot",
    "benchmark_id": benchmark["benchmark_id"],
    "benchmark_sha256": BENCHMARK_SHA256,
    "chunk_source": str(LEGAL_PATH),
    "embedding_model": EMBEDDING_MODEL,
    "dense_similarity": "cosine_via_normalized_dot_product",
    "lexical_retriever": "BM25Okapi",
    "fusion": "Reciprocal Rank Fusion",
    "rrf_k": RRF_K,
    "candidate_depth_per_retriever": CANDIDATE_DEPTH,
    "diversification": {
        "group": "file + article/location",
        "max_per_group": MAX_PER_SOURCE_GROUP
    },
    "top_k": TOP_K,
    "retrieval_method": "hybrid_dense_bm25_rrf_diverse_v1",
    "query_decomposition": False,
    "llm_used_in_retrieval": False,
    "iterative_retrieval": False
}

CONFIG_PATH.write_text(
    json.dumps(config, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Saved final frozen retrieval:")
print(" -", OUTPUT_PATH)
print(" -", CONFIG_PATH)


Saved final frozen retrieval:
 - data\retrieval_results_nb03_final_hybrid_static_rag.csv
 - data\retrieval_config_nb03_final_hybrid_static_rag.json


# Final Build02 definition

The final Build02 retriever is now frozen as:

> **Single-query hybrid static RAG**  
> Dense semantic retrieval + BM25 lexical retrieval  
> → Reciprocal Rank Fusion  
> → legal-source diversification  
> → fixed Top-5 evidence  
> → LLM answer generation in NB04

## Experimental boundary

NB03c multi-query retrieval remains an **ablation / exploratory experiment** and is not part of the final Build02 architecture.

The next notebook, NB04, should compare:

1. **Prompt-only**
2. **RAG using this frozen Top-5**

with the same LLM and the exact same Q1–Q10 benchmark.
